# Session 17: Calibration and multivariate models

A refinery or an export terminal decides whether a batch of diesel can be loaded on a cargo for Europe, where the minimum cetane number is 51.0, or is sold in the United States, where the minimum is 40. The cetane number is measured by an engine test, ASTM D613. This notebook asks whether a near-infrared (NIR) spectrum of the fuel can stand in for that test: it builds calibrations that predict cetane number from a spectrum and measures their error on fuels they were not fitted to. Section 1 works one wavelength, three wavelengths and every wavelength on Dataset A, 133 calibration fuels and 82 validation fuels. Section 2 applies a partial least squares (PLS) model to your fuel, with the choice of its size written by you. Section 3 compares the PLS call on your fuel with the three-band model's. Section 4 is the judgment cell.

The data are real: S. Hutzler, Southwest Research Institute (2004), NIR spectra and properties of diesel fuels, distributed by Eigenvector Research as `SWRI_Diesel_NIR_CSV.zip` on its [data sets page](https://eigenvector.com/resources/data-sets/), accessed 2026-11-10. The page states no licence, so the course does not copy the files; the first cell of section 1 downloads them into `data/`. `data/split.csv` gives each fuel's set, and `data/README.md` describes the files and who has which fuel.

**To submit.** Put your name in the next cell. Click **Restart** in the notebook toolbar, then **Run All**, check that every cell ran without an error, save, and upload `activity.ipynb` to the Session 17 assignment on Canvas by 1:30 pm on Thursday, December 3. Submissions are individual. Every student downloads the same files, checked byte for byte, so the graders' run reads the same data yours did. The notebook is scored 0 to 3: 1 if it runs from top to bottom after a restart and holds your prompt record from section 2, and 0 to 2 for the judgment cell, by the rubric in the syllabus, with the format in `ai-practices.md`.

*Name:*



## Setup and imports

The next cell imports `sys` and `Path` from the standard library, NumPy, pandas and Matplotlib, and `savgol_filter` and `find_peaks` from sessions 11 and 12, which locate the bands in section 1. Two things are new this session, from scikit-learn: `LinearRegression`, which fits a model with one coefficient for each input column, and `PLSRegression`, partial least squares regression. The cell prints nothing when it works.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, savgol_filter
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression

## Parameters

The data files and the rows where their headers end; the two cetane limits, the engine test's reproducibility and the factor that turns it into a standard deviation; the second-derivative filter and the smallest band it reports; the three bands, the single band and the collinear pair of section 1; how many latent variables TASK 2 tries; then your fuel, set in section 2. Leave `FUEL` as `CHANGE-ME` until section 2.

In [ ]:
DATA_DIR = Path("data")
SPEC_FILE = DATA_DIR / "diesel_spec.csv"     # the NIR spectra, one fuel per row; downloaded by data/fetch_diesel.py
PROP_FILE = DATA_DIR / "diesel_prop.csv"     # the same fuels' properties; CN is the cetane number by the ASTM D613 engine test
SPLIT_FILE = DATA_DIR / "split.csv"          # each fuel's set: calibration, validation or record; data/README.md
SPEC_AXIS_ROW = 9                            # the row of the spectra file holding the wavelengths, nm, labelled "Axisscale"; the spectra start on the next row
PROP_HEADER_ROW = 8                          # the row of the properties file holding the column names; the fuels start on the next row
LIMIT_EU = 51.0                              # minimum cetane number of diesel sold in the EU, Directive 2009/30/EC, Annex II: the cargo's limit
LIMIT_US = 40.0                              # minimum cetane number of No. 1-D and No. 2-D diesel, ASTM D975; section 1 checks the fuels against it
D613_REPRODUCIBILITY = (2.5, 3.3)            # cetane numbers: two laboratories' D613 results on one fuel differ by more than this about once in twenty, CN 44 to 56 (Van Gerpen, citing ASTM D613-1988)
R_PER_SD = 2.8                               # a reproducibility limit is 2.8, i.e. 1.96 times the square root of 2, times the standard deviation of one laboratory's result (ISO 5725); notes section 6
SAVGOL_WINDOW, SAVGOL_ORDER = 15, 3          # points and polynomial order of the second-derivative filter, session 11
BAND_MIN = 5e-4                              # absorbance per channel squared: the smallest peak of the negative second derivative reported as a band
ILS_NM = [998.0, 1042.0, 1064.0]             # nm, the three bands of the second derivative's first group, the ones ILS uses; data/README.md
ONE_NM = 1064.0                              # nm, the single band: of the three, the one most correlated with cetane
PAIR_NM = [1060.0, 1066.0]                   # nm, two neighboring channels on the 1064 nm band; section 1 shows they are collinear
LV_MAX = 15                                  # latent variables tried in TASK 2
FUEL = "CHANGE-ME"   # your fuel's label, from data/assignments.csv by your student number, section 2

## 1. Explore: one wavelength, three, or every one

Dataset A is two sets of fuels from the file: 133 **calibration** fuels, which every model is fitted to, and 82 **validation** fuels, which no model is fitted to and which measure each model's error. The question for this section is how many wavelengths a calibration for cetane needs. The first cell downloads the two files into `data/` and checks them, a second on campus Wi-Fi. It then reads both, checks the layout against what the files say about themselves, and prints the fuels in each set and their cetane numbers against the two limits.

In [ ]:
%run data/fetch_diesel.py
spec_raw = pd.read_csv(SPEC_FILE, header=None, low_memory=False).dropna(axis=1, how="all")   # the file's header rows hold text, so no header row is given
prop_raw = pd.read_csv(PROP_FILE, header=None, low_memory=False).dropna(axis=1, how="all")   # a trailing comma on every row makes an empty last column
assert spec_raw.iloc[SPEC_AXIS_ROW, 0] == "Axisscale", f"row {SPEC_AXIS_ROW} of {SPEC_FILE.name} is not the wavelength row"
wavelength = spec_raw.iloc[SPEC_AXIS_ROW, 2:].astype(float).to_numpy()      # nm
spectra = pd.DataFrame(spec_raw.iloc[SPEC_AXIS_ROW + 1:, 2:].astype(float).to_numpy(),
                       index=spec_raw.iloc[SPEC_AXIS_ROW + 1:, 1].astype(str).to_numpy(), columns=wavelength)
props = pd.DataFrame(prop_raw.iloc[PROP_HEADER_ROW + 1:, 2:].astype(float).to_numpy(),
                     index=prop_raw.iloc[PROP_HEADER_ROW + 1:, 1].astype(str).to_numpy(), columns=prop_raw.iloc[PROP_HEADER_ROW, 2:].to_list())
assert spectra.index.equals(props.index), "the two files list different fuels"
cetane = props["CN"]
split = pd.read_csv(SPLIT_FILE, dtype={"label": str}).set_index("label")["set"]
cal = split.index[split == "calibration"]
val = split.index[split == "validation"]
X_cal, y_cal = spectra.loc[cal].to_numpy(), cetane[cal].to_numpy()          # absorbance; cetane number
X_val, y_val = spectra.loc[val].to_numpy(), cetane[val].to_numpy()
print(f"spectra: {len(spectra)} fuels by {len(wavelength)} channels, {wavelength[0]:g} to {wavelength[-1]:g} nm every {np.diff(wavelength).mean():g} nm; "
      f"missing absorbances {int(spectra.isna().sum().sum())}")
print(f"properties: {', '.join(props.columns)}; fuels with a cetane number {int(cetane.notna().sum())} of {len(props)}")
print(f"data/split.csv: {split.value_counts().to_dict()}; every fuel in it has a cetane number: {bool(cetane[split.index].notna().all())}")
both = np.concatenate([y_cal, y_val])
print(f"cetane number, calibration and validation fuels: {both.min():.1f} to {both.max():.1f}, median {np.median(both):.1f}; "
      f"below {LIMIT_US:g}: {(both < LIMIT_US).sum()}; below {LIMIT_EU:g}: {(both < LIMIT_EU).sum()} of {len(both)}")

An NIR spectrum of diesel holds overtones and combinations of C-H stretching vibrations, so its bands carry the mix of aromatic, branched and straight-chain hydrocarbons that sets the cetane number. Overlapping bands blur into a few broad humps; the negative of the second derivative turns each band into a sharp peak. The next cell plots the calibration fuels' mean spectrum, the negative of its second derivative from `savgol_filter`, and the correlation of each channel's absorbance with cetane, with the bands in `ILS_NM` marked. It prints every band `find_peaks` reports and the correlation at each marked channel.

In [ ]:
mean_spectrum = X_cal.mean(axis=0)                                           # absorbance
neg_d2 = -savgol_filter(mean_spectrum, SAVGOL_WINDOW, SAVGOL_ORDER, deriv=2)   # absorbance per channel squared
bands, _ = find_peaks(neg_d2, height=BAND_MIN)
r = np.array([np.corrcoef(X_cal[:, j], y_cal)[0, 1] for j in range(len(wavelength))])
col = {w: int(np.flatnonzero(wavelength == w)[0]) for w in ILS_NM + PAIR_NM}   # the column of each named wavelength
print(f"bands of the negative second derivative above {BAND_MIN:g}: {', '.join(f'{wavelength[b]:g}' for b in bands)} nm")
print("correlation with cetane: " + ", ".join(f"{w:g} nm {r[col[w]]:+.2f}" for w in ILS_NM + PAIR_NM)
      + f"; largest {r.max():+.2f} at {wavelength[r.argmax()]:g} nm, most negative {r.min():+.2f} at {wavelength[r.argmin()]:g} nm")

fig, axes = plt.subplots(3, 1, sharex=True, figsize=(8, 8))
axes[0].plot(wavelength, mean_spectrum, color="black")
axes[0].set_ylabel("absorbance")
axes[0].set_title(f"Dataset A: the {len(cal)} calibration fuels")
axes[1].plot(wavelength, neg_d2, color="black")
axes[1].plot(wavelength[bands], neg_d2[bands], "o", color="black", markerfacecolor="none", label="bands found")
axes[1].set_ylim(-neg_d2[bands].max(), 1.5 * neg_d2[bands].max())   # the rise at the 1550 nm edge runs off the top
axes[1].set_ylabel("-d²A/dλ² (per channel²)")
axes[2].plot(wavelength, r, color="black")
axes[2].axhline(0, color="lightgray", linewidth=0.8)
axes[2].set_ylabel("r with cetane number")
axes[2].set_xlabel("wavelength (nm)")
for ax in axes:
    for w in ILS_NM:
        ax.axvline(w, color="gray", linestyle="--", linewidth=0.8)
axes[2].plot(PAIR_NM, r[[col[w] for w in PAIR_NM]], "s", color="black", markerfacecolor="none", label="PAIR_NM")
axes[1].legend(fontsize="small")
axes[2].legend(fontsize="small")
fig.tight_layout()
plt.show()

The mean spectrum of the 133 calibration fuels has two groups of bands and rises steeply at the 1550 nm edge; the negative second derivative resolves the groups into six bands, at 998, 1042 and 1064 nm and at 1240, 1264 and 1288 nm. The correlation with cetane is -0.43 at 998 nm, +0.06 at 1042 nm and +0.69 at 1064 nm, close to its largest, +0.69 at 1066 nm, and its most negative value is -0.51 at 1534 nm.

**Inverse least squares (ILS)** regresses the cetane number on the absorbances at a few chosen wavelengths, one coefficient each, without needing the spectrum of any pure component. The next cell defines `rmse`, the root-mean-square difference between predicted and engine values, and fits three ILS models on the calibration fuels: the single band `ONE_NM`, the three bands `ILS_NM`, and `PAIR_NM`, two neighboring channels on the 1064 nm band. For each it prints the calibration error (RMSEC), the validation error and the coefficients, then the correlation between the two channels of the pair.

In [ ]:
def rmse(predicted, engine):
    """Root-mean-square difference between predicted and engine cetane numbers, in cetane numbers."""
    return float(np.sqrt(np.mean((np.ravel(predicted) - np.ravel(engine)) ** 2)))


ils_models = {}
for name, nms in (("one band", [ONE_NM]), ("ILS, three bands", ILS_NM), ("ILS, the pair", PAIR_NM)):
    cols = [int(np.flatnonzero(wavelength == w)[0]) for w in nms]
    model = LinearRegression().fit(X_cal[:, cols], y_cal)
    ils_models[name] = (model, cols)
    print(f"{name:17s} {', '.join(f'{w:g}' for w in nms):18s} nm: RMSEC {rmse(model.predict(X_cal[:, cols]), y_cal):.2f}, "
          f"validation {rmse(model.predict(X_val[:, cols]), y_val):.2f}; coefficients "
          + ", ".join(f"{c:+.0f}" for c in model.coef_) + " per absorbance unit")
pair_r = np.corrcoef(X_cal[:, col[PAIR_NM[0]]], X_cal[:, col[PAIR_NM[1]]])[0, 1]
print(f"correlation between the absorbances at {PAIR_NM[0]:g} and {PAIR_NM[1]:g} nm over the calibration fuels: {pair_r:.3f}")
ils, ils_cols = ils_models["ILS, three bands"]
val_ils = rmse(ils.predict(X_val[:, ils_cols]), y_val)                    # cetane numbers, section 3 uses it

If three wavelengths help, why not all 401? The next cell fits `LinearRegression` to every channel at once, ILS with 401 coefficients and 133 calibration fuels to fit them, and sets its RMSEC and validation error beside the three ILS models'. It then plots each validation fuel's predicted cetane number against its engine value, for the three-band ILS model and the 401-channel model, with the line where the two agree and the 51 limit on both axes.

In [ ]:
every = LinearRegression().fit(X_cal, y_cal)
rows = [(name, len(cols), rmse(m.predict(X_cal[:, cols]), y_cal), rmse(m.predict(X_val[:, cols]), y_val)) for name, (m, cols) in ils_models.items()]
rows.append(("every channel", X_cal.shape[1], rmse(every.predict(X_cal), y_cal), rmse(every.predict(X_val), y_val)))
table = pd.DataFrame(rows, columns=["model", "channels", "RMSEC", "validation"]).set_index("model")
print(f"errors in cetane numbers; {len(cal)} calibration fuels, {len(val)} validation fuels")
print(table.to_string(float_format=lambda v: f"{v:.3f}"))
print(f"ranked by RMSEC, lowest first: {'; '.join(table['RMSEC'].sort_values().index)}")
print(f"ranked by validation error, lowest first: {'; '.join(table['validation'].sort_values().index)}")
pred_ils_val, pred_every_val = ils.predict(X_val[:, ils_cols]), every.predict(X_val)
worst = int(np.argmax(np.abs(pred_every_val - y_val)))
print(f"validation fuels: engine {y_val.min():.1f} to {y_val.max():.1f}; ILS, three bands, predicts {pred_ils_val.min():.1f} to {pred_ils_val.max():.1f}; "
      f"every channel {pred_every_val.min():.1f} to {pred_every_val.max():.1f}, with {int((np.abs(pred_every_val - y_val) > 5).sum())} fuels more than 5 off "
      f"and its worst, engine {y_val[worst]:.1f}, predicted {pred_every_val[worst]:.1f}")

fig, ax = plt.subplots()
ax.plot(y_val, pred_ils_val, "o", color="black", markerfacecolor="none", label=f"ILS, three bands ({table.loc['ILS, three bands', 'validation']:.2f})")
ax.plot(y_val, pred_every_val, "s", color="gray", markerfacecolor="none", label=f"every channel ({table.loc['every channel', 'validation']:.2f})")
lims = [min(y_val.min(), pred_every_val.min()) - 1, max(y_val.max(), pred_every_val.max()) + 1]
ax.plot(lims, lims, color="black", linewidth=0.8, label="predicted = engine")
ax.axhline(LIMIT_EU, color="gray", linestyle=":", label=f"{LIMIT_EU:g}, the cargo's limit")
ax.axvline(LIMIT_EU, color="gray", linestyle=":")
ax.set_xlabel("engine (D613) cetane number")
ax.set_ylabel("predicted cetane number")
ax.set_title(f"Dataset A: the {len(val)} validation fuels")
ax.legend(fontsize="small")
fig.tight_layout()
plt.show()

On the 82 validation fuels, whose engine values run from 40.3 to 61.3, the three-band ILS predictions run from 41.2 to 59.4 and stay near the line, whereas the every-channel predictions run from 23.7 to 66.6, with 9 fuels more than 5 cetane numbers off and one fuel at 44.7 by engine predicted at 23.7; most fuels of both models sit below 51 on both axes.

**Explain.** In two or three sentences: the every-channel model has the lowest calibration error of the four and the highest validation error. What about its channels and its 133 fuels lets it fit the calibration fuels exactly, and what in the output shows that the fit does not carry to fuels it has not seen? Say what the pair's coefficients and correlation show about two channels that rise and fall together.

*Your answer:*

## 2. Apply: your fuel against 51

Dataset B is one fuel, named in `data/assignments.csv`, kept out of the calibration fuels and out of the validation fuels: no model in this notebook is fitted to it, and no choice is made on it. Section 1's sets carry over. What changes is the model. **PLS** regression uses every channel without the collapse section 1 showed: it builds a few **latent variables**, weighted sums of the 401 absorbances chosen for how strongly they vary with cetane, and regresses cetane on those. The number of latent variables, not the number of channels, sets how flexible the model is, and you choose it. The question for your fuel is which side of 51 its prediction falls on, and by how much more than the model's error.

**[TASK 1]** Find your student number in `data/assignments.csv` and set `FUEL` in the Parameters cell to the label beside it, for example `"21663"`, which is student 27's. Run the Parameters cell, then the cell under this paragraph. It checks that your fuel is a record, out of both of section 1's sets, and sets `x_fuel`, its spectrum as one row of 401 absorbances. Until `FUEL` is set, it stops with `KeyError: "no fuel labelled 'CHANGE-ME' in data/split.csv; set FUEL in Parameters from data/assignments.csv"`.

In [ ]:
if FUEL not in split.index:
    raise KeyError(f"no fuel labelled {FUEL!r} in data/split.csv; set FUEL in Parameters from data/assignments.csv")
x_fuel = spectra.loc[[FUEL]].to_numpy()        # one row, absorbance at each of the 401 channels
print(f"fuel {FUEL}: set {split[FUEL]!r} in data/split.csv; among the calibration fuels: {FUEL in cal}; among the validation fuels: {FUEL in val}; "
      f"spectrum of {x_fuel.shape[1]} absorbances, {x_fuel.min():.3f} to {x_fuel.max():.3f}")

**[TASK 2]** In a code cell you add under this one (**+ Code**), fit `PLSRegression(a, scale=False)` to the calibration fuels, `X_cal` and `y_cal`, for each number of latent variables `a` from 1 to `LV_MAX`; `scale=False` keeps each channel in absorbance, centered on its mean. For each model compute two errors with section 1's `rmse`, as its ILS cell did: on the calibration fuels, and on the validation fuels, `X_val` and `y_val`. Set `rmsec_by_lv` and `val_by_lv`, each a list of `LV_MAX` errors in cetane numbers, the first for one latent variable, and `N_LV`, the number of latent variables you choose, by this session's rule. The cell after it plots both curves and prints the validation error at your `N_LV` beside the lowest. It stops with `NameError: name 'N_LV' is not defined` until your cell has run. If you chose where the calibration error is lowest, it says so.

In [ ]:
print(f"N_LV = {N_LV}")
rmsec_by_lv, val_by_lv = np.asarray(rmsec_by_lv, dtype=float), np.asarray(val_by_lv, dtype=float)
assert len(rmsec_by_lv) == LV_MAX and len(val_by_lv) == LV_MAX, f"rmsec_by_lv and val_by_lv need {LV_MAX} errors each, one for each number of latent variables"
lv = np.arange(1, LV_MAX + 1)
lowest = int(lv[np.argmin(val_by_lv)])
if N_LV == int(lv[np.argmin(rmsec_by_lv)]) and (np.diff(rmsec_by_lv) < 0).all():
    print("N_LV is where RMSEC is lowest; RMSEC falls at every step, so that rule picks the most latent variables tried")
check = rmse(PLSRegression(N_LV, scale=False).fit(X_cal, y_cal).predict(X_val), y_val)
if abs(check - val_by_lv[N_LV - 1]) > 1e-9:
    print(f"val_by_lv gives {val_by_lv[N_LV - 1]:.3f} at N_LV, but PLS with {N_LV} latent variables, scale=False, fitted to X_cal and y_cal, "
          f"gives {check:.3f} on X_val and y_val: check which fuels each error uses, and that every model has scale=False")
print(f"validation error at N_LV = {N_LV}: {val_by_lv[N_LV - 1]:.2f} cetane numbers; lowest {val_by_lv.min():.2f}, at {lowest}; "
      f"the choice of {N_LV} costs {val_by_lv[N_LV - 1] - val_by_lv.min():.2f} on the validation fuels")
print("latent variables:  " + " ".join(f"{a:5d}" for a in lv))
print("RMSEC:             " + " ".join(f"{e:5.2f}" for e in rmsec_by_lv))
print("validation error:  " + " ".join(f"{e:5.2f}" for e in val_by_lv))

fig, ax = plt.subplots()
ax.plot(lv, rmsec_by_lv, "o-", color="black", markerfacecolor="none", label=f"calibration, {len(cal)} fuels (RMSEC)")
ax.plot(lv, val_by_lv, "s--", color="black", label=f"validation, {len(val)} fuels")
ax.axvline(N_LV, color="gray", linestyle=":", label=f"N_LV = {N_LV}")
ax.set_xlabel("latent variables (count)")
ax.set_ylabel("error (cetane number)")
ax.set_title("PLS on Dataset A: error against the number of latent variables")
ax.legend()
fig.tight_layout()
plt.show()

*Write your one sentence description of the plot here, with where each curve is lowest.*

**[TASK 3]** Fill the two lines of the next cell: `pls`, a PLS model with `N_LV` latent variables, `scale=False`, fitted to the calibration fuels alone, and `pred_pls`, its predicted cetane number for `x_fuel`. `predict` returns an array with one prediction per fuel, so for one fuel it holds one value, and `pred_pls` is that value as a number. The cell then checks how many fuels the model was fitted to and that it is mean centered, and prints your prediction, the engine value of your fuel and the side of 51 each falls on. Until the lines are filled it stops with `TypeError: float() argument must be a string or a real number, not 'ellipsis'`.

In [ ]:
pls = PLSRegression(...)     # [TASK 3] N_LV latent variables, scale=False, then .fit() to the calibration fuels only
pred_pls = float(...)       # [TASK 3] pls's prediction for x_fuel: predict returns an array of one value; pred_pls is that value, cetane number

if pls.x_scores_.shape[0] != len(cal):
    raise ValueError(f"pls was fitted to {pls.x_scores_.shape[0]} fuels, not the {len(cal)} calibration fuels: fit it to X_cal and y_cal alone, "
                     "so that your fuel and the validation fuels stay out of it")
assert pls.n_components == N_LV, f"pls has {pls.n_components} latent variables, not N_LV = {N_LV}"
if pls.scale:
    raise ValueError("pls was fitted with scale=True, which autoscales every channel; this session's model is PLSRegression(N_LV, scale=False), mean centered")
err_pls = val_by_lv[N_LV - 1]                 # cetane numbers, the validation error at N_LV
engine = float(cetane[FUEL])                  # cetane number, ASTM D613
side = lambda v: "at or above" if v >= LIMIT_EU else "below"   # noqa: E731
print(f"fuel {FUEL}: PLS with {N_LV} latent variables predicts {pred_pls:.2f}, {side(pred_pls)} {LIMIT_EU:g}, "
      f"{abs(pred_pls - LIMIT_EU):.2f} from it, or {abs(pred_pls - LIMIT_EU) / err_pls:.2f} times the validation error of {err_pls:.2f}")
print(f"engine (D613) value: {engine:.1f}, {side(engine)} {LIMIT_EU:g}; prediction minus engine value {pred_pls - engine:+.2f}")

**Explain.** Before you compare: in two sentences, say which side of 51 your PLS prediction puts your fuel on and how many validation errors it sits from 51, and whether the engine value agrees. Then compare with a neighbor: set your PLS prediction, engine value and call beside your neighbor's. Add one sentence on where your two answers differ and what in your data explains it. If no neighbor is ready, notes section 8 gives the solution's fuel.

*Your answer:*

**Ask the assistant** whether your fuel meets the cargo's limit, in your own words. This is the first half of today's AI practice. Save the notebook first. Add a markdown cell under this one (**+ Markdown**) holding the lines below, and write your prompt on the first before you send it. Ask whether your fuel, by its label, is at or above 51 and how sure that call is, and say where the data are: this session's notebook and its `data/` folder. Read the answer before you act on it, and note anything in it you would want done differently every time. If the assistant asks to run Python with `uv run`, click **Allow**. If it asks to edit this notebook, click **Decline**; the answer in the chat is what you read.

```
**Prompt record**

- **My prompt:**
- **The line I wrote in CLAUDE.local.md, and what in the first answer it changes:**
- **What the second answer did, against what section 3 prints:**
```

**Keep a CLAUDE.md**, the second half of today's AI practice. In VS Code's Explorer, open `CLAUDE.local.md` in the top folder of your clone of the course repository, the one that holds `CLAUDE.md` and `README.md`. If you kept the file from session 10, add a line under your old one; if not, right-click that top folder, choose **New File...**, and name it `CLAUDE.local.md`. Write one line, your own, from something the first answer did that you would want done differently every time. Write it so that the next answer either visibly follows it or does not; notes section 7 says what makes a line work. Save the file, and copy the line onto the second line of your prompt record, with what in the first answer it changes.

In the Claude Code prompt box, type `/` and choose **Clear conversation**, so the assistant starts again and reads the file. If **Clear conversation** is not in the menu, open the Command Palette (**View**, then **Command Palette**) and run **Claude Code: Open in New Tab**. Then send the same prompt as before. Keep the second answer open: you fill the third line of your record in section 3, against what its first cell prints.

## 3. Analyze: the three-band model or PLS

Both models are defensible for the terminal. The three-band ILS model has three coefficients a chemist can read against the spectrum and check on a new instrument; PLS uses every channel and has the lower validation error. The question for this section is whether the choice changes the call on your fuel. The next cell predicts your fuel with section 1's three-band model, prints each model's prediction with plus and minus its own validation error and the side of 51 it falls on, and sets both beside the engine value and the D613 reproducibility.

In [ ]:
pred_ils = float(ils.predict(x_fuel[:, ils_cols])[0])          # cetane number
calls = pd.DataFrame({"prediction": [pred_ils, pred_pls], "validation error": [val_ils, err_pls]},
                     index=["ILS, three bands", f"PLS, {N_LV} latent variables"])
calls["low"] = calls["prediction"] - calls["validation error"]
calls["high"] = calls["prediction"] + calls["validation error"]
calls["call at 51"] = [("pass" if v >= LIMIT_EU else "fail") for v in calls["prediction"]]
calls["51 inside low to high"] = (calls["low"] <= LIMIT_EU) & (LIMIT_EU <= calls["high"])
calls["minus engine"] = calls["prediction"] - engine
print(f"fuel {FUEL}, cetane numbers; engine (D613) value {engine:.1f}, call {'pass' if engine >= LIMIT_EU else 'fail'}")
print(calls.to_string(float_format=lambda v: f"{v:.2f}"))
sd_d613 = [r / R_PER_SD for r in D613_REPRODUCIBILITY]           # cetane numbers, one laboratory's standard deviation
print(f"the two models {'agree' if calls['call at 51'].nunique() == 1 else 'disagree'} at {LIMIT_EU:g}; "
      f"D613 reproducibility {D613_REPRODUCIBILITY[0]:g} to {D613_REPRODUCIBILITY[1]:g}: the engine value is {abs(engine - LIMIT_EU):.1f} from the limit")
print(f"as a standard deviation, the kind of number a validation error is: D613 {sd_d613[0]:.2f} to {sd_d613[1]:.2f} between laboratories; "
      f"PLS {err_pls:.2f}, ILS {val_ils:.2f}")

fig, ax = plt.subplots()
for y, (name, row) in zip((2, 1), calls.iterrows()):
    ax.errorbar(row["prediction"], y, xerr=row["validation error"], fmt="o", color="black", markerfacecolor="none", capsize=4)
ax.plot(engine, 0, "s", color="black", label="engine (D613) value")
ax.axvline(LIMIT_EU, color="gray", linestyle="--", label=f"{LIMIT_EU:g}, the cargo's limit")
ax.set_yticks([2, 1, 0], [*calls.index, "engine, D613"])
ax.set_ylim(-0.7, 2.7)
ax.set_xlabel("cetane number")
ax.set_title(f"Fuel {FUEL}: predictions with their validation errors")
ax.legend(fontsize="small", loc="lower right")
fig.tight_layout()
plt.show()

*Write your one sentence description of the plot here.*

Read the second answer from the practice beside the first, against what the first cell of this section printed. Did it follow your line? Read the numbers it quotes against what the first cell of this section printed. Write that on the third line of your prompt record. A line that did not visibly change the answer is a finding too: say so, and what you would write instead. Your prompt record is one candidate for part (b) of the judgment cell.

**Explain.** In two or three sentences, say which model's call you would sign for your fuel and why, from the two predictions and their validation errors: is either prediction more than its own error from 51, and is the engine value more than the D613 reproducibility from it? If the two models disagree, say which channels section 1 says the three-band model reads and what PLS adds. This is the evidence part (a) of the judgment cell needs.

*Your answer:*

## 4. Judgment cell

The cell below is scored 0 to 2 by the rubric in the syllabus, in the format that `ai-practices.md` gives, and you write it yourself. Part (a): as the terminal's analyst, does your batch go on the European cargo at 51.0? Cite your PLS prediction and its validation error, the three-band prediction, your fuel's engine value and the D613 reproducibility, and say what a wrong call would cost the seller: a cargo below 51 rejected or treated at the receiving terminal, or a fuel that met 51 sold into the 40 market. Part (b): one judgment entry, a suggestion from the assistant that you changed, rejected or verified, and the reason from your notebook. Your prompt record from section 2 is the natural source; the reason has to cite a value from your output, not a general rule. This session's AI practice, named in the notes, is *Keep a CLAUDE.md*, and an entry that came from using it is the expected one, though any entry grounded in your data counts.

**Judgment cell**

**(a)**

**(b) Judgment entry**

- **What the assistant suggested:**
- **What I did:**
- **Why, from my data:** 

## Software environment

The last cell prints the Python and package versions this notebook ran with, so a result can always be traced to the software that produced it.

In [ ]:
import matplotlib
import scipy
import sklearn

print(f"Python       {sys.version.split()[0]}")
print(f"NumPy        {np.__version__}")
print(f"pandas       {pd.__version__}")
print(f"SciPy        {scipy.__version__}")
print(f"Matplotlib   {matplotlib.__version__}")
print(f"scikit-learn {sklearn.__version__}")